In [17]:
import numpy as np
import pandas as pd
import random
import joblib
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional, GlobalAveragePooling1D, BatchNormalization, Conv1D, MaxPooling1D, Attention, Masking, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
import os
import tensorflow as tf

# --- TIME-AWARE FEATURE EXTRACTOR ---
def build_features(df):
    df = df.copy()
    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)

    # Calculate Time Delta
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)

    df['accel_x'] = df.groupby('track_no')['vel_x'].diff() / df['dt']
    df['accel_y'] = df.groupby('track_no')['vel_y'].diff() / df['dt']
    df['accel_z'] = df.groupby('track_no')['vel_z'].diff() / df['dt']
    df[['accel_x', 'accel_y', 'accel_z']] = df[['accel_x', 'accel_y', 'accel_z']].fillna(0)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1>0 and n2>0 and dt[i]>0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else: res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).std().reset_index(level=0, drop=True)

    def calc_dir_change(group):
        h = group['heading'].values
        res = [0.0]
        for i in range(1, len(h)):
            delta = abs((h[i] - h[i-1]) % 360)
            res.append(1.0 if min(delta, 360-delta) > 45 else 0.0)
        return np.array(res, dtype=np.float64)

    df['direction_change_rate'] = df.groupby('track_no').apply(calc_dir_change).reset_index(level=0, drop=True)
    df['direction_change_rate'] = pd.to_numeric(df['direction_change_rate'], errors='coerce').fillna(0.0)

    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    return df.fillna(0.0)

# --- ADVANCED TRAINING SIMULATOR (Identical to your stress test) ---
def generate_complex_trajectory(track_class, duration=80, dt=10.0):
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    track_id = random.randint(100000, 999999)

    spawn_radius = random.uniform(10000, 80000)
    spawn_angle = random.uniform(0, 2 * np.pi)
    pos_x = spawn_radius * np.cos(spawn_angle)
    pos_y = spawn_radius * np.sin(spawn_angle)

    if track_class == "CIVILIAN":
        pos_z = np.random.uniform(5000, 12000)
        v_mag = np.random.uniform(220, 310)
    elif track_class == "MISSILE":
        pos_z = np.random.uniform(50, 500) if random.random() < 0.3 else np.random.uniform(8000, 22000)
        v_mag = np.random.uniform(400, 1100)
    elif track_class == "HELICOPTER":
        pos_z = np.random.uniform(20, 1500)
        v_mag = np.random.uniform(0, 85)
    elif track_class == "FIGHTER_JET":
        pos_z = np.random.uniform(500, 19000)
        v_mag = np.random.uniform(250, 750)
    elif track_class == "SKIFF":
        pos_z = 0.0
        v_mag = np.random.uniform(2, 28)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    start_time = 3991182426
    actual_elapsed_time = 0.0

    maneuver_timer = 0
    maneuver_type = "NONE"
    target_turn_rate = 0.0
    in_burst_loss = False

    for t in range(duration):
        if maneuver_timer <= 0:
            if track_class == "FIGHTER_JET" and random.random() < 0.15:
                maneuver_timer = random.randint(4, 10)
                maneuver_type = random.choice(["EVASIVE_TURN", "SPLIT_S_DIVE"])
                target_turn_rate = np.radians(random.uniform(-15, 15))
            elif track_class == "CIVILIAN" and random.random() < 0.10:
                maneuver_timer = random.randint(15, 30)
                maneuver_type = "HOLDING_PATTERN"
                target_turn_rate = np.radians(random.choice([-3, 3]))
            elif track_class == "HELICOPTER" and random.random() < 0.20:
                maneuver_timer = random.randint(3, 8)
                maneuver_type = random.choice(["POP_UP", "SUDDEN_HOVER"])
            elif track_class == "SKIFF" and random.random() < 0.25:
                maneuver_timer = random.randint(2, 6)
                maneuver_type = "ZIG_ZAG"
                target_turn_rate = np.radians(random.uniform(-25, 25))
            elif track_class == "MISSILE" and random.random() < 0.10:
                maneuver_timer = random.randint(3, 5)
                maneuver_type = "TERMINAL_POP_UP"
            else:
                maneuver_type = "CRUISE"

        if maneuver_timer > 0:
            maneuver_timer -= 1
            if maneuver_type in ["EVASIVE_TURN", "HOLDING_PATTERN", "ZIG_ZAG"]:
                c, s = np.cos(target_turn_rate * dt), np.sin(target_turn_rate * dt)
                vel[0], vel[1] = vel[0]*c - vel[1]*s, vel[0]*s + vel[1]*c
            elif maneuver_type == "SPLIT_S_DIVE":
                vel[2] -= 40 * dt
                vel[0] *= 1.05; vel[1] *= 1.05
            elif maneuver_type == "POP_UP":
                vel[2] = 25
                vel[0] *= 0.5; vel[1] *= 0.5
            elif maneuver_type == "SUDDEN_HOVER":
                vel *= 0.5
            elif maneuver_type == "TERMINAL_POP_UP":
                vel[2] = 50
        else:
            if track_class != "CIVILIAN":
                vel += np.random.normal(0, 1.5, 3)
            else:
                vel += np.random.normal(0, 0.2, 3)
                vel[2] *= 0.95

        if track_class == "SKIFF": vel[2] = 0.0

        pos_x += vel[0] * dt
        pos_y += vel[1] * dt
        pos_z += vel[2] * dt

        if pos_z < 0 and track_class != "SKIFF":
            pos_z = 0.0
            vel[2] = 0.0
        if track_class == "SKIFF": pos_z = 0.0

        actual_elapsed_time += dt

        if in_burst_loss:
            if random.random() < 0.60: in_burst_loss = False
            else: continue
        else:
            if random.random() < 0.08:
                in_burst_loss = True
                continue

        r_mag = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
        noise_multiplier = max(1.0, (r_mag / 25000) ** 1.5)

        pos_x_noisy = pos_x + np.random.normal(0, 15 * noise_multiplier)
        pos_y_noisy = pos_y + np.random.normal(0, 15 * noise_multiplier)
        if track_class == "SKIFF": pos_z_noisy = abs(np.random.normal(0, 8 * noise_multiplier))
        else: pos_z_noisy = pos_z + np.random.normal(0, 20 * noise_multiplier)

        vel_x_noisy = vel[0] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_y_noisy = vel[1] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_z_noisy = vel[2] + np.random.normal(0, 1.5 * noise_multiplier)

        timestamp = start_time + int(actual_elapsed_time)

        pos_x_list.append(pos_x_noisy); pos_y_list.append(pos_y_noisy); pos_z_list.append(pos_z_noisy)
        vel_x_list.append(vel_x_noisy); vel_y_list.append(vel_y_noisy); vel_z_list.append(vel_z_noisy)
        track_no_list.append(track_id); track_class_list.append(track_class); ntp_sec_list.append(timestamp)

    df_data = {
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    }
    return pd.DataFrame(df_data)

def generate_dataset(num_samples=250, duration=80):
    dfs = []
    track_classes = ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]
    for class_choice in track_classes:
        for _ in range(num_samples):
            df = generate_complex_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 5:
                dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# --- ARCHITECTURE ---
def build_robust_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    masked = Masking(mask_value=0.0)(inputs)

    conv1 = Conv1D(32, kernel_size=3, activation='relu', padding='same')(masked)
    conv2 = Conv1D(32, kernel_size=5, activation='relu', padding='same')(masked)
    conv3 = Conv1D(32, kernel_size=7, activation='relu', padding='same')(masked)

    merged = concatenate([conv1, conv2, conv3])
    merged = BatchNormalization()(merged)
    merged = MaxPooling1D(pool_size=2)(merged)

    lstm_out = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2))(merged)

    attention = Attention()([lstm_out, lstm_out])
    attention_pooled = GlobalAveragePooling1D()(attention)

    second_lstm = Bidirectional(LSTM(32, return_sequences=False))(lstm_out)

    combined = concatenate([attention_pooled, second_lstm])
    dense = Dense(64, activation='relu')(combined)
    dense = BatchNormalization()(dense)
    dense = Dropout(0.4)(dense)

    outputs = Dense(num_classes, activation='softmax')(dense)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

if __name__ == "__main__":
    if not os.path.exists("trained_model"): os.makedirs("trained_model")
    print("Generating complex real-world training data...")
    df = generate_dataset(num_samples=300) # Generating 1500 total tracks for better generalization
    print("Building memory features...")
    df_enhanced = build_features(df)

    # CRITICAL ADDITION: 'dt' is now explicitly passed to the network
    feature_cols = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    scaler = StandardScaler()
    df_enhanced[feature_cols] = scaler.fit_transform(df_enhanced[feature_cols])
    joblib.dump(scaler, "trained_model/feature_scaler.pkl")

    seq_length = 50
    track_sequences, track_labels = [], []
    print("Building progressive sequences...")

    for track_id, group in df_enhanced.groupby('track_no'):
        data_matrix = group.sort_values('ntp_sec')[feature_cols].values

        for i in range(1, len(data_matrix) + 1):
            if i <= seq_length:
                window = data_matrix[:i]
                padded = np.pad(window, ((seq_length - len(window), 0), (0, 0)), 'constant')
                track_sequences.append(padded)
            else:
                window = data_matrix[i-seq_length:i]
                track_sequences.append(window)
            track_labels.append(group['track_class'].iloc[0])

    X, y = np.array(track_sequences), np.array(track_labels)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    joblib.dump(le, "trained_model/label_encoder.pkl")

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
    model = build_robust_model((X.shape[1], X.shape[2]), len(le.classes_))

    callbacks = [EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
                 ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)]

    print("Training Next-Level Model...")
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=64, callbacks=callbacks)
    model.save("trained_model/radar_target_classifier.keras")
    print("Training Complete!")

Generating complex real-world training data...
Building memory features...
Building progressive sequences...
Training Next-Level Model...
Epoch 1/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 55s 38ms/step - accuracy: 0.9406 - loss: 0.1521 - val_accuracy: 0.9701 - val_loss: 0.0702 - learning_rate: 0.0010
Epoch 2/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 51s 38ms/step - accuracy: 0.9703 - loss: 0.0729 - val_accuracy: 0.9774 - val_loss: 0.0504 - learning_rate: 0.0010
Epoch 3/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 47s 36ms/step - accuracy: 0.9758 - loss: 0.0578 - val_accuracy: 0.9839 - val_loss: 0.0375 - learning_rate: 0.0010
Epoch 4/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.9793 - loss: 0.0510 - val_accuracy: 0.9846 - val_loss: 0.0347 - learning_rate: 0.0010
Epoch 5/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.9811 - loss: 0.0462 - val_accuracy: 0.9862 - val_loss: 0.0323 - learning_rate: 0.0010
Epoch 6/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.9827 - loss:

2

In [18]:
import numpy as np
import pandas as pd
import random

def generate_complex_trajectory(track_class, duration=60, dt=10.0):
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    track_id = random.randint(100000, 999999)

    # Spawn targets at varying ranges to test range-dependent noise
    spawn_radius = random.uniform(10000, 80000)
    spawn_angle = random.uniform(0, 2 * np.pi)
    pos_x = spawn_radius * np.cos(spawn_angle)
    pos_y = spawn_radius * np.sin(spawn_angle)

    # Overlapping boundary initializations
    if track_class == "CIVILIAN":
        pos_z = np.random.uniform(5000, 12000)
        v_mag = np.random.uniform(220, 310)
    elif track_class == "MISSILE":
        # 30% chance to be a "Sea Skimmer" missile (very low altitude)
        pos_z = np.random.uniform(50, 500) if random.random() < 0.3 else np.random.uniform(8000, 22000)
        v_mag = np.random.uniform(400, 1100)
    elif track_class == "HELICOPTER":
        pos_z = np.random.uniform(20, 1500)
        v_mag = np.random.uniform(0, 85) # Can start in a hover (0 m/s)
    elif track_class == "FIGHTER_JET":
        pos_z = np.random.uniform(500, 19000)
        v_mag = np.random.uniform(250, 750)
    elif track_class == "SKIFF":
        pos_z = 0.0
        v_mag = np.random.uniform(2, 28)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    start_time = 3991182426
    actual_elapsed_time = 0.0

    # State Machine Variables for Sustained Maneuvers
    maneuver_timer = 0
    maneuver_type = "NONE"
    target_turn_rate = 0.0

    # Sensor failure simulation variables
    in_burst_loss = False

    for t in range(duration):
        # ---------------------------------------------------
        # 1. SUSTAINED MANEUVER LOGIC (The "Real World" aspect)
        # ---------------------------------------------------
        if maneuver_timer <= 0:
            # Chance to start a new maneuver
            if track_class == "FIGHTER_JET" and random.random() < 0.15:
                maneuver_timer = random.randint(4, 10) # Lasts 40 to 100 seconds
                maneuver_type = random.choice(["EVASIVE_TURN", "SPLIT_S_DIVE"])
                target_turn_rate = np.radians(random.uniform(-15, 15)) # Deg per second

            elif track_class == "CIVILIAN" and random.random() < 0.10:
                maneuver_timer = random.randint(15, 30) # Long holding pattern
                maneuver_type = "HOLDING_PATTERN"
                target_turn_rate = np.radians(random.choice([-3, 3])) # Standard rate turn

            elif track_class == "HELICOPTER" and random.random() < 0.20:
                maneuver_timer = random.randint(3, 8)
                maneuver_type = random.choice(["POP_UP", "SUDDEN_HOVER"])

            elif track_class == "SKIFF" and random.random() < 0.25:
                maneuver_timer = random.randint(2, 6)
                maneuver_type = "ZIG_ZAG"
                target_turn_rate = np.radians(random.uniform(-25, 25))

            elif track_class == "MISSILE" and random.random() < 0.10:
                maneuver_timer = random.randint(3, 5)
                maneuver_type = "TERMINAL_POP_UP" # Pops up before diving
            else:
                maneuver_type = "CRUISE"

        # Execute active maneuver
        if maneuver_timer > 0:
            maneuver_timer -= 1

            if maneuver_type in ["EVASIVE_TURN", "HOLDING_PATTERN", "ZIG_ZAG"]:
                # Apply sustained turning over dt
                c, s = np.cos(target_turn_rate * dt), np.sin(target_turn_rate * dt)
                vel[0], vel[1] = vel[0]*c - vel[1]*s, vel[0]*s + vel[1]*c

            elif maneuver_type == "SPLIT_S_DIVE":
                vel[2] -= 40 * dt # Massive sustained dive
                vel[0] *= 1.05; vel[1] *= 1.05 # Accelerate in dive

            elif maneuver_type == "POP_UP":
                vel[2] = 25 # Rapid climb
                vel[0] *= 0.5; vel[1] *= 0.5 # Lose horizontal speed

            elif maneuver_type == "SUDDEN_HOVER":
                vel *= 0.5 # Bleed speed rapidly to 0

            elif maneuver_type == "TERMINAL_POP_UP":
                vel[2] = 50 # Sharp climb

        else:
            # Normal Cruising - Add slight natural drift
            if track_class != "CIVILIAN":
                vel += np.random.normal(0, 1.5, 3)
            else:
                vel += np.random.normal(0, 0.2, 3)
                vel[2] *= 0.95 # Civil flights tend to strictly self-level

        # Strict Boundaries
        if track_class == "SKIFF":
            vel[2] = 0.0

        # ---------------------------------------------------
        # 2. PHYSICS UPDATE
        # ---------------------------------------------------
        pos_x += vel[0] * dt
        pos_y += vel[1] * dt
        pos_z += vel[2] * dt

        # Hard ground floor
        if pos_z < 0 and track_class != "SKIFF":
            pos_z = 0.0
            vel[2] = 0.0
        if track_class == "SKIFF": pos_z = 0.0

        actual_elapsed_time += dt

        # ---------------------------------------------------
        # 3. BURST PACKET LOSS SIMULATION
        # ---------------------------------------------------
        if in_burst_loss:
            if random.random() < 0.60: # 60% chance to recover radar lock
                in_burst_loss = False
            else:
                continue # Skip appending! Model receives a massive time gap next packet.
        else:
            if random.random() < 0.08: # 8% chance to lose radar lock
                in_burst_loss = True
                continue

        # ---------------------------------------------------
        # 4. RANGE-DEPENDENT SENSOR NOISE
        # ---------------------------------------------------
        r_mag = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
        noise_multiplier = max(1.0, (r_mag / 25000) ** 1.5) # Noise scales exponentially past 25km

        pos_x_noisy = pos_x + np.random.normal(0, 15 * noise_multiplier)
        pos_y_noisy = pos_y + np.random.normal(0, 15 * noise_multiplier)

        if track_class == "SKIFF":
            # Sea clutter / wave bounce
            pos_z_noisy = abs(np.random.normal(0, 8 * noise_multiplier))
        else:
            pos_z_noisy = pos_z + np.random.normal(0, 20 * noise_multiplier)

        vel_x_noisy = vel[0] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_y_noisy = vel[1] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_z_noisy = vel[2] + np.random.normal(0, 1.5 * noise_multiplier)

        # ---------------------------------------------------
        # 5. RECORD DATA
        # ---------------------------------------------------
        timestamp = start_time + int(actual_elapsed_time)

        pos_x_list.append(pos_x_noisy)
        pos_y_list.append(pos_y_noisy)
        pos_z_list.append(pos_z_noisy)
        vel_x_list.append(vel_x_noisy)
        vel_y_list.append(vel_y_noisy)
        vel_z_list.append(vel_z_noisy)
        track_no_list.append(track_id)
        track_class_list.append(track_class)
        ntp_sec_list.append(timestamp)

    # Note: Because of burst drop, the lists might be shorter than `duration`. That's intended!
    df_data = {
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    }
    return pd.DataFrame(df_data)

def generate_stress_test_dataset(num_samples=150, duration=80):
    dfs = []
    track_classes = ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]
    samples_per_class = num_samples // len(track_classes)

    for class_choice in track_classes:
        for _ in range(samples_per_class):
            df = generate_complex_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 5: # Only keep if at least 5 packets survived the burst loss
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

if __name__ == "__main__":
    print("Initializing Real-World Radar Stress Test Generator...")
    # Generating 150 tracks, tracked over 80 potential scans (800 seconds max)
    test_df = generate_stress_test_dataset(num_samples=150, duration=80)
    test_df.to_csv("test_radar_data.csv", index=False)
    print(f"Successfully generated advanced stress-test dataset with {len(test_df)} packets.")
    print("Features injected:")
    print(" - Burst packet loss (Time jumps > 10s)")
    # Print max dt to prove time jumps exist
    test_df['dt'] = test_df.groupby('track_no')['ntp_sec'].diff()
    max_jump = test_df['dt'].max()
    print(f" - Maximum sensor blackout period detected: {max_jump} seconds")
    print(" - Range-dependent exponentially scaling noise")
    print(" - Sustained Holding Patterns, Evasive Dives, and Hovering Maneuvers")

Initializing Real-World Radar Stress Test Generator...
Successfully generated advanced stress-test dataset with 10623 packets.
Features injected:
 - Burst packet loss (Time jumps > 10s)
 - Maximum sensor blackout period detected: 110.0 seconds
 - Range-dependent exponentially scaling noise
 - Sustained Holding Patterns, Evasive Dives, and Hovering Maneuvers


3

In [19]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib

def build_features(df):
    df = df.copy()
    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)

    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)

    df['accel_x'] = df.groupby('track_no')['vel_x'].diff() / df['dt']
    df['accel_y'] = df.groupby('track_no')['vel_y'].diff() / df['dt']
    df['accel_z'] = df.groupby('track_no')['vel_z'].diff() / df['dt']
    df[['accel_x', 'accel_y', 'accel_z']] = df[['accel_x', 'accel_y', 'accel_z']].fillna(0)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1>0 and n2>0 and dt[i]>0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else: res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).std().reset_index(level=0, drop=True)

    def calc_dir_change(group):
        h = group['heading'].values
        res = [0.0]
        for i in range(1, len(h)):
            delta = abs((h[i] - h[i-1]) % 360)
            res.append(1.0 if min(delta, 360-delta) > 45 else 0.0)
        return np.array(res, dtype=np.float64)

    df['direction_change_rate'] = df.groupby('track_no').apply(calc_dir_change).reset_index(level=0, drop=True)
    df['direction_change_rate'] = pd.to_numeric(df['direction_change_rate'], errors='coerce').fillna(0.0)

    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    return df.fillna(0.0)

def predict_single_track(track_group, model, le, scaler):
    feature_cols = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    try:
        features = track_group[feature_cols].values
        features_scaled = scaler.transform(features)

        seq_length = 50
        if len(features_scaled) >= seq_length:
            input_data = features_scaled[-seq_length:].reshape(1, seq_length, -1)
        else:
            padded = np.pad(features_scaled, ((seq_length - len(features_scaled), 0), (0, 0)), 'constant')
            input_data = padded.reshape(1, seq_length, -1)

        pred_probs = model.predict(input_data, verbose=0)[0]
        predicted_class_idx = np.argmax(pred_probs)
        confidence = np.max(pred_probs)
        predicted_label = le.inverse_transform([predicted_class_idx])[0]

        return {'prediction': predicted_label, 'confidence': float(confidence)}
    except Exception as e:
        return {'prediction': f'ERROR: {str(e)}', 'confidence': 0.0}

def simulate_streaming_radar(df_test, model, le, scaler):
    print("\n" + "="*60)
    print(" REAL-TIME STREAMING SIMULATION (Packet-by-Packet)")
    print("="*60)

    sample_tracks = df_test['track_no'].unique()[:3]
    for tid in sample_tracks:
        track_data = df_test[df_test['track_no'] == tid].sort_values('ntp_sec')
        actual = track_data['track_class'].iloc[0]
        print(f"\n📡 Tracking ID {tid} (Actual Target: {actual})")

        checkpoints = [1, 3, 10, 25, len(track_data)]
        for i in checkpoints:
            if i > len(track_data): continue

            current_stream = track_data.iloc[:i]
            res = predict_single_track(current_stream, model, le, scaler)

            status = "✅" if res['prediction'] == actual else "⏳"
            print(f"  Packets received: {i:2d} -> Predicted: {res['prediction']:12s} | Confidence: {res['confidence']:.2%} {status}")

def process_test_file(file_path):
    print("Loading trained model...")
    model = load_model("trained_model/radar_target_classifier.keras", compile=False)
    le = joblib.load("trained_model/label_encoder.pkl")
    scaler = joblib.load("trained_model/feature_scaler.pkl")
    print("✓ Model and encoders loaded successfully")

    df_test = pd.read_csv(file_path).dropna(subset=['track_no'])
    print("Building cumulative memory features...")
    df_test = build_features(df_test)

    simulate_streaming_radar(df_test, model, le, scaler)

    print("\nProcessing final full sequences for report...")
    final_results = []
    for track_id, group in df_test.groupby('track_no'):
        actual_label = group['track_class'].iloc[0]
        pred_res = predict_single_track(group, model, le, scaler)
        final_results.append({
            'Track ID': track_id, 'Actual Class': actual_label, 'Predicted Class': pred_res['prediction'],
            'Confidence': f"{pred_res['confidence']:.4f}", 'Result': '✅ Correct' if actual_label == pred_res['prediction'] else '❌ Wrong'
        })
    return final_results

def visualize_results_table(results):
    df_res = pd.DataFrame(results)
    correct_count = len(df_res[df_res['Result'] == '✅ Correct'])
    total = len(df_res)
    accuracy = (correct_count / total) * 100 if total > 0 else 0

    print("\n" + "="*60)
    print(f" FINAL CLASSIFICATION REPORT ")
    print("="*60)
    print(f"Total Tracks: {total} | Correct: {correct_count} | Accuracy: {accuracy:.2f}%")
    print("-" * 60)
    print(df_res.to_string(index=False))
    print("="*60)

if __name__ == "__main__":
    results_list = process_test_file("test_radar_data.csv")
    if results_list: visualize_results_table(results_list)

Loading trained model...
✓ Model and encoders loaded successfully
Building cumulative memory features...

 REAL-TIME STREAMING SIMULATION (Packet-by-Packet)

📡 Tracking ID 109048 (Actual Target: MISSILE)
  Packets received:  1 -> Predicted: MISSILE      | Confidence: 99.83% ✅
  Packets received:  3 -> Predicted: MISSILE      | Confidence: 100.00% ✅
  Packets received: 10 -> Predicted: MISSILE      | Confidence: 100.00% ✅
  Packets received: 25 -> Predicted: MISSILE      | Confidence: 100.00% ✅
  Packets received: 73 -> Predicted: MISSILE      | Confidence: 100.00% ✅

📡 Tracking ID 114785 (Actual Target: HELICOPTER)
  Packets received:  1 -> Predicted: HELICOPTER   | Confidence: 96.29% ✅
  Packets received:  3 -> Predicted: HELICOPTER   | Confidence: 98.72% ✅
  Packets received: 10 -> Predicted: HELICOPTER   | Confidence: 99.70% ✅
  Packets received: 25 -> Predicted: HELICOPTER   | Confidence: 100.00% ✅
  Packets received: 73 -> Predicted: HELICOPTER   | Confidence: 100.00% ✅

📡 Trackin